# PilotNet: architecture, training, and validation

This notebook studies the 2016 NVIDIA camera-to-steering policy. It uses the repository implementation rather than a notebook-only copy, so every experiment exercises the same code as `train.py` and `eval.py`.

## 1. The learning problem

Behavioral cloning learns a scalar steering command from a front-facing image. A supervised sample is `(image, steering)`, and the training objective is mean squared error. This predicts the expert action on recorded states; it does not by itself solve the covariate shift that occurs after the learned policy makes a mistake.

In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from pilotnet.data import DrivingDataset
from pilotnet.engine import evaluate, train_epoch
from pilotnet.models import PilotNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 2. Input and normalization

PilotNet consumes RGB images shaped `(3, 66, 200)`. The dataset resizes source images to this geometry and returns floating-point pixels in `[0, 1]`. The model performs `image - 0.5` as its first operation. Keeping normalization in the model makes the inference contract explicit and prevents a training-serving mismatch.

## 3. Convolutional feature extractor

The first three convolutions use `5x5`, stride-2 kernels with 24, 36, and 48 channels. They rapidly reduce spatial resolution while expanding feature capacity. Two `3x3`, stride-1 layers with 64 channels refine the road and scene features. ELU follows every convolution; it retains negative responses smoothly instead of zeroing them as ReLU would.

## 4. Steering regressor

For a `66x200` input, the feature extractor produces `64x1x18` activations. Flattening yields 1,152 values, followed by fully connected widths `100 -> 50 -> 10 -> 1`. The final layer is linear because steering is a signed continuous value, not a probability or class.

In [ ]:
model = PilotNet().to(device)
example = torch.rand(2, 3, 66, 200, device=device)
prediction = model(example)
print(model)
print('Prediction shape:', tuple(prediction.shape))
print('Parameters:', sum(parameter.numel() for parameter in model.parameters()))

## 5. Driving-log dataset

Create separate CSV files for training and validation. Each must have `image_path` and `steering` columns; paths are relative to that CSV. The training dataset horizontally flips images at random and negates steering, which is the only augmentation included here because it has an exact geometric target transformation. Keep route- or time-adjacent frames in the same split to avoid leakage.

In [ ]:
train_csv = Path('../data/train.csv')
val_csv = Path('../data/val.csv')

train_dataset = DrivingDataset(train_csv, augment=True)
val_dataset = DrivingDataset(val_csv)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=device.type == 'cuda')
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=device.type == 'cuda')
images, steering = next(iter(train_loader))
print(images.shape, steering.shape)

## 6. Optimization

MSE gives larger steering mistakes disproportionately more weight, so sharp turns can dominate the loss. The accompanying MAE reports the typical absolute error in the dataset's steering unit. Validation runs without augmentation and must guide checkpoint selection; training MSE alone is not a measure of generalization.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
history = []
for epoch in range(1, 6):
    train_metrics = train_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate(model, val_loader, device)
    history.append({'epoch': epoch, **train_metrics, **{f'val_{key}': value for key, value in val_metrics.items()}})
    print(history[-1])

## 7. Validation and next experiments

Save the model selected by validation MSE, then inspect errors by steering magnitude, lighting, road type, and recovery situations. Offline metrics cannot establish closed-loop safety. Natural follow-up experiments are balanced steering sampling, photometric augmentation, camera recovery data, and saliency analysis, each evaluated against the unchanged split.

In [ ]:
checkpoint_path = Path('../artifacts/notebook_best.pt')
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({'model_state_dict': model.state_dict(), 'history': history}, checkpoint_path)
final_metrics = evaluate(model, val_loader, device)
print('Saved:', checkpoint_path)
final_metrics